In [8]:
import os
import sys

sys.path.insert(0, os.path.abspath("src"))

In [2]:
%pwd

'c:\\Users\\hp\\Text-Summarization\\research'

In [3]:
os.chdir("..")


In [4]:
%pwd

'c:\\Users\\hp\\Text-Summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: list


In [9]:
from text.constants import *
from text.utils.common import read_yaml, create_directories
from text.entity.config_entity import DataIngestionConfig

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            ALL_REQUIRED_FILES=config.ALL_REQUIRED_FILES,
        )

        return data_validation_config

In [12]:
import os
from text.logging import logger 

In [14]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_files_exist(self) -> bool:
        try:
            validation_status = True

            all_files = os.listdir(
                os.path.join("artifacts", "data_ingestion", "samsum_dataset")
            )

            for file in self.config.ALL_REQUIRED_FILES:
                if file not in all_files:
                    validation_status = False
                    break

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            logger.exception(e)
            raise e

In [16]:
try:
    config = ConfigurationManager()

    data_validation_config = config.get_data_validation_config()

    data_validation = DataValidation(config=data_validation_config)

    validation_status = data_validation.validate_all_files_exist()

    print("Validation Status:", validation_status)

except Exception as e:
    raise e

[2026-07-16 16:38:47,796: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-16 16:38:47,800: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-16 16:38:47,805: INFO: common: Directory created: artifacts]
[2026-07-16 16:38:47,810: INFO: common: Directory created: artifacts/data_validation]
Validation Status: True
